# 02 — Training (D3–D6)

**YOLOv8s / YOLO11s / YOLO26s under identical settings.**

The paper's entire claim is *identical conditions*. Everything comes from `src/config.py`. **Do not tune per model** — a different lr for one model kills the controlled comparison and the contribution with it.

**Measured on T4 (smoke test, 2026-07-20):** 54.3 s/epoch from mounted storage → ~36 min per model, **~1.8 h for three**. Staging locally (§0.3) should roughly halve that.

---
### ⚠️ Accelerator: `GPU T4 x2` — NOT P100

P100 is **sm_60**; current PyTorch requires **sm_70+**. It fails or silently falls back to CPU. Select **`GPU T4 x2`** (sm_75), keep `device=0`.

**Don't attempt 2×T4 DDP.** Kaggle bills by wall-clock not per-GPU, so `T4 x2` with `device=0` costs the same quota. Notebook DDP spawn failures are a known sink. T4 has FP16 tensor cores but **no bf16** — `amp=True`, never `bf16`.

---
### Prerequisite — attach notebook 01's output

1. In **notebook 01**: Save Version → **"Save & Run All (Commit)"**, *not* Quick Save. Push to GitHub first — the commit re-clones.
2. Here: **Add Data** → **"Notebook Output Files"** → filter **"Your Work"** → notebook 01.

Mounts at `/kaggle/input/notebooks/<username>/<slug>/vindr_yolo/`. `C.find_data_yaml()` searches shallow-first with a recursive fallback, and repairs a stale absolute `path:` if the dataset was built before that fix.

In [ ]:
!pip install -q ultralytics wandb

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd

from src import config as C
from src.training import train as T
from src.utils.run_logger import RunLogger, benchmark_inference, capture_gpu

!cd {REPO} && git rev-parse --short HEAD

In [ ]:
# strict=True -> raises immediately on sm_60 (P100) rather than after a session
# of confusing CPU-speed epochs.
gpu = capture_gpu(strict=True)
gpu

In [ ]:
DATA_YAML = C.find_data_yaml()
# Fallback if the search ever misses -- same validation runs:
# DATA_YAML = C.find_data_yaml(
#     "/kaggle/input/notebooks/<username>/01-data-prep/vindr_yolo/data.yaml")

### 0.3 Stage the dataset locally — do this

Measured from the mounted dataset:

```
Slow image access detected (read: 35.1±19.3 MB/s)
Cache directory ... is not writable, cache not saved
```

3,515 images × 216 KB ≈ **760 MB per epoch**. At 35 MB/s that's ~22 s of a 54 s epoch spent on I/O, and Ultralytics re-scans labels (~15 s) every run because it can't cache to a read-only mount.

One-off ~1 min copy. Over 3 models × 40 epochs it pays back many times.

In [ ]:
DATA_YAML = C.stage_dataset_local(DATA_YAML)
DATA_ROOT = DATA_YAML.parent
print(DATA_YAML.read_text())

## 1. W&B overhead measurement (D3, ~10 min)

5 epochs with and without W&B, compare `sec_per_epoch` from `RunLogger`.

General figure is 1–3% (async background process; **image logging is the expensive part, not scalars**). Measure it here rather than trusting that. If >5%, disable image logging and keep the scalars.

Also re-measures epoch time **after local staging** — compare against the 54.3 s/epoch baseline from mounted storage.

In [ ]:
_ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=False,
                runs_dir=C.RUNS_DIR / "smoke_nowandb")

In [ ]:
# Needs WANDB_API_KEY in Add-ons -> Secrets.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    have_wandb = True
except Exception as e:
    print(f"no W&B secret ({e}) -- continuing without. RunLogger still records everything.")
    have_wandb = False

if have_wandb:
    _ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=True,
                    runs_dir=C.RUNS_DIR / "smoke_wandb")

In [ ]:
a = RunLogger.master_table(C.RUNS_DIR / "smoke_nowandb").sec_per_epoch.iloc[-1]
print(f"local staged: {a:.1f}s/epoch   (mounted baseline was 54.3)")
print(f"  -> {a*C.EPOCHS/60:.0f} min per model, {a*C.EPOCHS*3/3600:.1f} h for three")

if have_wandb:
    b = RunLogger.master_table(C.RUNS_DIR / "smoke_wandb").sec_per_epoch.iloc[-1]
    print(f"with wandb:   {b:.1f}s/epoch  (overhead {(b/a-1)*100:.1f}%)")

total_h = a * C.EPOCHS * 3 / 3600
if total_h > 3:
    print(f"\n⚠ {total_h:.1f} h projected. Consider 2 models "
          f"(yolov8s + yolo26s = Axis B, NMS vs NMS-free).")
    print("  Cut a MODEL, not epochs -- protocol consistency beats model count.")

## 2. Full training runs (D3–D5)

Sequential, one GPU. **Checkpoints save every epoch** — the 12 h session cap will interrupt you; re-run with `resume=True` on the affected model.

Two models instead of three: `T.train_all(DATA_YAML, models=["yolov8s", "yolo26s"])`.

In [ ]:
results = T.train_all(DATA_YAML, use_wandb=have_wandb)
{k: v["status"] for k, v in results.items()}

In [ ]:
# Session killed mid-run? Resume just that model:
# _ = T.train_one("yolo26s", DATA_YAML, resume=True)

## 3. mAP@0.4 — the competition metric

> ⚠️ **Ultralytics cannot give you mAP@0.4.** It reports @0.5 and @0.5:0.95 only. VinBigData used IoU > 0.4. If @0.5 silently lands in a column labelled @0.4, the related-work comparison misaligns and the error survives to camera-ready.

> [!CAUTION]
> **Our numbers are not directly comparable to the published ones.**
>
> We train and evaluate on the **positive-only subset** — every image contains at least one finding, so the model never has an opportunity to false-positive on a healthy chest. That inflates precision and mAP relative to a full-set evaluation.
>
> Evidence: the 5-epoch smoke test already reached **mAP@0.5 = 0.310** against YOLO-CXR's published **0.338** at full training. A 5-epoch model does not nearly match a published architecture — the protocols differ.
>
> Whether the published works used positive-only is **not established**. Keep them in a *related work* table with an explicit protocol column; never in the same column as ours. State the protocol in the abstract, not just methods.

In [ ]:
from ultralytics import YOLO
from src.evaluation import metrics as M

weights = T.all_best_weights()
evals = {}
for key, w in weights.items():
    print(f"\n=== {key} ===")
    evals[key] = M.evaluate_full(
        YOLO(w), DATA_ROOT / "images" / "test", DATA_ROOT / "labels" / "test",
        imgsz=C.IMGSZ,
    )

In [ ]:
summary = pd.DataFrame({
    k: {"mAP@0.4": v["map40"], "mAP@0.5": v["map50"]} for k, v in evals.items()
}).T.round(4)
print(summary)

print("\nPublished — DIFFERENT PROTOCOL, related-work table only:")
for name, vals in C.PUBLISHED_BASELINES.items():
    print(f"  {name}: {vals}")

### ⛔ Gate D4

Reference reached ~0.45 CV positive-only with EfficientDet at 896–1024px, 50 epochs. You're at 512px, `s`-scale, 40 epochs.

- **0.25–0.40** — reasonable, proceed
- **< 0.20** — labels or fusion are broken, not the model. Stop.
- **> 0.50** — check the split for leakage before believing it.

In [ ]:
m = evals["yolov8s"]["map40"]
if m < 0.20:
    print(f"⛔ GATE FAILED: mAP@0.4={m:.4f} < 0.20. Debug the data pipeline.")
elif m > 0.50:
    print(f"⚠ mAP@0.4={m:.4f} unexpectedly high. Check the split for leakage.")
else:
    print(f"✓ gate passed: mAP@0.4={m:.4f}")

### Per-class — where the score actually comes from

Smoke test (5 epochs, yolov8s) showed the headline mAP is carried by two classes:

| | mAP@0.5 |
|---|---|
| Aortic enlargement | **0.945** |
| Cardiomegaly | **0.912** |
| *other 12, mean* | **0.207** |
| all 14 | 0.310 |

Both leaders are **anatomically anchored** — the aorta and heart sit in the same place in every image, so the model can score by learning position rather than pathology. Strip them and real abnormality detection is at ~0.21.

That is a finding, not a defect, and it sets up **Axis A** directly: Calcification 0.097 and Other lesion 0.045 against 0.945 for a fixed-location structure. If explanation faithfulness tracks that gap, the paper's central claim has strong evidence.

**Report per-class throughout.** A single mAP hides the entire story.

In [ ]:
pc = pd.DataFrame({k: v["detail_40"]["per_class"] for k, v in evals.items()}).round(4)
pc["n_gt"] = pd.Series(list(evals.values())[0]["detail_40"]["n_gt"])
pc = pc.sort_values("n_gt")
print(pc)

# How much of mAP is the two anchored classes?
anchored = ["Aortic enlargement", "Cardiomegaly"]
for k in evals:
    per = pd.Series(evals[k]["detail_40"]["per_class"]).dropna()
    print(f"{k}: all-14 {per.mean():.4f} | "
          f"excl. anchored {per.drop(anchored, errors='ignore').mean():.4f}")

> **Pneumothorax (n≈10) will look degenerate.** The smoke test gave `P=1.000, R=0.000` — one near-miss detection and nothing else. That is what AP on ten boxes looks like. The reporting rule was fixed in notebook 01 before any score was seen: keep all 14 in mAP, show `n`, caveat n<30 in limitations.

## 4. Inference benchmark (D10) — same card, one session

> ⚠️ **mAP is hardware-independent. Latency is not.**
>
> YOLO26 is marketed as edge-first (claimed 43% CPU speed gain over YOLO11), so a latency table is expected. Valid only if produced **here** — all checkpoints back to back, one session, one card — not stitched from training runs that may have landed on different GPUs.

In [ ]:
RunLogger.check_hardware_consistency(C.RUNS_DIR)
bench = benchmark_inference(weights, imgsz=C.IMGSZ, out_dir=str(C.RUNS_DIR))
pd.DataFrame(bench).T[["ms_per_image", "fps", "n_params_M", "gpu_family"]]

## 5. Persist

`/kaggle/working` dies with the session. Commit via **Save Version → "Save & Run All"** so notebook 03 can attach these weights.

In [ ]:
import json
(C.RUNS_DIR / "eval_summary.json").write_text(json.dumps(
    {k: {"map40": v["map40"], "map50": v["map50"],
         "per_class_40": v["detail_40"]["per_class"],
         "n_gt": v["detail_40"]["n_gt"]} for k, v in evals.items()},
    indent=2, default=str))

print("weights:")
for k, w in weights.items():
    print(f"  {k}: {w}")
print("\nNext: Save Version -> 'Save & Run All (Commit)', then attach to notebook 03.")